<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/PCA_from_MD_trajectories.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# PCA from MD trajectories using MDAnalysis
# ============================================

!pip install MDAnalysis MDAnalysisData matplotlib numpy pandas

import MDAnalysis as mda
from MDAnalysis.analysis import pca
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import files

# ============================================
# 1️⃣ Upload dos arquivos (estrutura + trajetória)
# ============================================
print("📂 Envie os arquivos de estrutura e trajetória (.pdb/.gro e .xtc/.trr)")
uploaded = files.upload()

top_file = [f for f in uploaded if f.endswith(('.pdb', '.gro'))][0]
traj_file = [f for f in uploaded if f.endswith(('.xtc', '.trr'))][0]

u = mda.Universe(top_file, traj_file)

# ============================================
# 2️⃣ Seleção dos átomos (ex: Cα)
# ============================================
protein = u.select_atoms("protein and name CA")
print(f"✅ Selecionados {len(protein)} átomos Cα para PCA.")

# ============================================
# 3️⃣ PCA (usando coordenadas centradas)
# ============================================
pca_analysis = pca.PCA(u, select='protein and name CA', align=True, mean=None)
pca_analysis.run()

# Autovalores e autovetores
eigenvalues = pca_analysis.results.eigenvalues
eigenvectors = pca_analysis.results.eigenvectors  # shape: (n_modes, 3*N_atoms)

# Projeção de cada frame nos PCs
proj = pca_analysis.transform(protein).T  # frames × PCs

# ============================================
# 4️⃣ Salvando resultados
# ============================================
np.savetxt("PCA_eigenvectors.txt", eigenvectors)
np.savetxt("PCA_eigenvalues.txt", eigenvalues)
np.savetxt("PCA_projection.txt", proj)

print("💾 Arquivos salvos: PCA_eigenvectors.txt, PCA_eigenvalues.txt, PCA_projection.txt")

# ============================================
# 5️⃣ Variância explicada (gráfico tipo painel A)
# ============================================
var_frac = eigenvalues / np.sum(eigenvalues)
cum_var = np.cumsum(var_frac)

plt.figure(figsize=(6,4))
plt.bar(range(1, 21), var_frac[:20], color='royalblue')
plt.plot(range(1, 21), cum_var[:20], 'o-', color='royalblue')
plt.axhline(0.8, ls='--', color='k')
plt.axvline(np.argmax(cum_var>0.8)+1, ls='--', color='k')
plt.xlabel('PCA index')
plt.ylabel('Fraction of variance')
plt.title('Fraction of total variance (MD PCA)')
plt.show()
